In [0]:
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("merge_keys", "")
dbutils.widgets.text("load_type", "")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")


In [0]:
bronze_df = spark.read.table(f"adb_caresync.bronze.{table_name}").filter(f"landing_timestamp = '{landing_timestamp}'")
print(f"Bronze df read from table adb_caresync.bronze.{table_name} at timestamp {landing_timestamp}")

In [0]:
import pyspark.sql.functions as F
# insert timestamp will be updated with new inserted timestamp, update_timestamp is used only to record the update query timestamp  
updated_df = bronze_df.withColumn("insert_timestamp", F.current_timestamp()).withColumn("update_timestamp", F.current_timestamp())


In [0]:
target_table_name = f"adb_caresync.silver.{table_name}"

In [0]:
# 3. Handle write operation according to load_type
if load_type == "FULL":
    print(f"[INFO] Initiating FULL load: Overwriting existing data in '{target_table_name}'...")
    (
        updated_df.write.mode("overwrite").saveAsTable(target_table_name)       
    )
    print(f"[SUCCESS] Table '{target_table_name}' successfully overwritten.")
elif load_type == "APPEND":
    print(f"[INFO] Initiating APPEND load: Appending records into '{target_table_name}'...")
    (
        updated_df.write.mode("append").saveAsTable(target_table_name)
    )
    print(f"[SUCCESS] Records successfully appended to '{target_table_name}'.")

elif load_type == "MERGE":
    # Ensure the target table exists before running SQL MERGE
    if not spark.catalog.tableExists(target_table_name):
        print(f"[WARN] Target table '{target_table_name}' does not exist. Initializing via overwrite...")
        (
            updated_df.write
            .mode("overwrite")
            .saveAsTable(target_table_name)
        )
        print(f"[SUCCESS] Target table '{target_table_name}' created.")
    else:
        # Register source dataframe as a temporary view for SQL access
        temp_view_name = f"source_{table_name}_view"
        updated_df.createOrReplaceTempView(temp_view_name)

        # Build join condition (supports single or comma-separated composite keys)
        keys = [k.strip() for k in merge_keys.split(",") if k.strip()]
        join_condition = " AND ".join([f"target.{k} = source.{k}" for k in keys])

        # Exclude 'insert_timestamp' so the original insertion timestamp is preserved on updates
        update_set_clause = ",\n        ".join([
            f"target.{col} = source.{col}" 
            for col in updated_df.columns 
            if col != "insert_timestamp"
        ])

        # Build explicit INSERT column and value lists
        insert_columns = ", ".join(updated_df.columns)
        insert_values = ", ".join([f"source.{col}" for col in updated_df.columns])

        # Execute Delta MERGE via standard SQL
        merge_sql = f"""
        MERGE INTO {target_table_name} AS target
        USING {temp_view_name} AS source
        ON {join_condition}
        WHEN MATCHED THEN
          UPDATE SET
            {update_set_clause}
        WHEN NOT MATCHED THEN
          INSERT ({insert_columns})
          VALUES ({insert_values})
        """
        
        spark.sql(merge_sql)
        print(f"[SUCCESS] MERGE operation completed successfully on '{target_table_name}'.")

else:
    raise ValueError(f"Unsupported load_type '{load_type}'. Expected 'APPEND' or 'MERGE' or 'FULL'.")

In [0]:
record_count = updated_df.count()
print("Processed Record Count: " + str(record_count))
dbutils.notebook.exit(str(record_count))